# GraphAllocBench Sweeps (Quick Start)

Minimal guide to launch hyperparameter and architecture sweeps using the packaged utilities.

You can either:
- Use the Python API inside this notebook, or
- Run the CLI equivalent: `python -m graphallocbench.train_utils.sweep --sweep --env_name problem_0 --sweep_config_path sweep_config/hyperparam_search.yml`

Proceed in order below.

## 1. Setup and Imports
Import packages and ensure availability of torch-geometric & wandb.

In [ ]:
# Minimal imports for sweeps with robust path handling
import os, sys, yaml, random
from pathlib import Path
import numpy as np
import torch

# Ensure graphallocbench importable (works even if not pip-installed yet)
try:
    import graphallocbench  # noqa: F401
except ModuleNotFoundError:
    probe = Path.cwd()
    for p in [probe] + list(probe.parents):
        if (p / 'graphallocbench' / '__init__.py').exists():
            sys.path.insert(0, str(p))
            break
    import graphallocbench  # noqa: F401

from graphallocbench.train_utils import sweep_all, sweep_arch_all, set_seeds

import wandb
print('Torch', torch.__version__, 'CUDA:', torch.cuda.is_available())

### W&B API Key Setup
To authenticate W&B choose one method BEFORE running sweeps:
1. CLI login (recommended):
```
!wandb login
```
2. Environment variable (non-interactive):
```
%env WANDB_API_KEY=your_key_here
```
3. Programmatic:
```
import wandb, os
os.environ['WANDB_API_KEY'] = 'your_key_here'
wandb.login(key=os.environ['WANDB_API_KEY'])
```
Validate with:
```
import wandb; print('W&B logged in:', wandb.login())
```
If offline desired: set `%env WANDB_MODE=offline`.


## 2. Project Path Configuration

In [ ]:
# Robust package root discovery
from pathlib import Path

def find_pkg_root():
    start = Path.cwd()
    for p in [start] + list(start.parents):
        # Case: repo root containing package
        if (p / 'graphallocbench' / 'config' / 'problems').is_dir():
            return p / 'graphallocbench'
        # Case: already inside package directory
        if (p / 'config' / 'problems').is_dir() and (p / '__init__.py').exists():
            return p
    raise FileNotFoundError('Could not locate graphallocbench package root')

PKG_ROOT = find_pkg_root()
CONFIG_DIR = PKG_ROOT / 'config' / 'problems'
SWEEP_CFG = (PKG_ROOT.parent / 'sweep_config' / 'hyperparam_search.yml') if (PKG_ROOT.parent / 'sweep_config').exists() else (PKG_ROOT / 'sweep_config' / 'hyperparam_search.yml')

print('Package root:', PKG_ROOT)
print('Config exists:', CONFIG_DIR.exists())
print('Sweep config exists:', SWEEP_CFG.exists())
print('Num problems:', len(list(CONFIG_DIR.glob('problem_*.yml'))))

## 3. Load Problem Configurations

In [ ]:
# Quick peek at available problems
problems = sorted(p.stem for p in CONFIG_DIR.glob('problem_*.yml'))
problems[:10]

['problem_0',
 'problem_1a',
 'problem_1b',
 'problem_1c',
 'problem_2a',
 'problem_2b',
 'problem_2c',
 'problem_3a',
 'problem_3b',
 'problem_4a']

## 4. Launch a Hyperparameter Sweep (Single Environment)

This creates (or reuses) a W&B sweep and spawns agents for one environment.

Parameters are defined in `sweep_config/hyperparam_search.yml`. Ensure you are logged into W&B first.

In [ ]:
# Example: 2 agents for problem_0
# Uncomment to run
sweep_all({'problem_0': 2}, sweep_config_path=str(SWEEP_CFG), project_name='GraphAllocBench-v3')

## 5. Launch Architecture Sweep

Runs the sweep for all registered architectures across chosen environments.

In [ ]:
# Example: 1 agent per architecture for problem_0 (adjust project name if desired)
sweep_arch_all({'problem_0': 1}, sweep_config_path=str(SWEEP_CFG), project_name='GraphAllocBench-GNN-v2')

## 6. CLI Usage (Alternative)
Instead of the Python API, you can start an agent process directly:

```
# Hyperparameter sweep agent
python -m graphallocbench.train_utils.sweep --sweep --env_name problem_0 --sweep_config_path sweep_config/hyperparam_search.yml

# Architecture sweep agent (specify architecture index)
python -m graphallocbench.train_utils.sweep --sweep_arch --env_name problem_0 --arch_idx 0 --sweep_config_path sweep_config/hyperparam_search.yml
```

Ensure environment variables `WANDB_PROJECT` (and optionally `WANDB_ENTITY`) are set, or pass `project_name` via the higher-level helpers.